# Fine-tuning a model with the Trainer API

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
%%capture
!pip install datasets evaluate transformers accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [1]:
from datasets import load_dataset  # Import the HuggingFace function to load built-in datasets
from transformers import AutoTokenizer, DataCollatorWithPadding  # Import classes for tokenization and dynamic padding

raw_datasets = load_dataset("glue", "mrpc")  # Load the MRPC subset of the GLUE benchmark as a dataset dictionary
checkpoint = "bert-base-uncased"  # Define the name of the pre-trained checkpoint to use (BERT base, uncased)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)  # Load the corresponding tokenizer for the above checkpoint

def tokenize_function(example):
    # Tokenize both sentence1 and sentence2 per example and enable truncation to max model length
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

# Tokenize all examples in the dataset using batching for efficiency, returning a new dataset dictionary
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
# Create a data collator that automatically pads batches to the longest sequence in the batch at runtime
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [ ]:
# The following lines set up the training configuration for the Trainer.
from transformers import TrainingArguments  # Import the TrainingArguments class, used to specify training settings.

# Create a TrainingArguments object. 
# The string "test-trainer" specifies the output directory where model predictions and checkpoints will be saved.
training_args = TrainingArguments("test-trainer", num_train_epochs=10)  # Train for 10 epochs

In [4]:
# Import the class to easily load a pre-trained model for sequence classification tasks.
from transformers import AutoModelForSequenceClassification

# Load a BERT model pre-trained on general text and prepare it for sequence classification (such as sentence pairs).
# We specify 'num_labels=2' because the MRPC task is a binary classification problem (are the sentences paraphrases?).
# If this is the first time running, this line will download the pre-trained weights; otherwise, it loads them from cache.
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Import the Trainer class from the transformers library, which provides an easy API to train (fine-tune) and evaluate models.
from transformers import Trainer

# Instantiate the Trainer object. 
# This object handles the training loop, evaluation, and other related utilities.
trainer = Trainer(
    model,  # The model to be trained (in this case, a BERT sequence classification model).
    training_args,  # The configuration containing training hyperparameters like output directory, number of epochs, etc.
    train_dataset=tokenized_datasets["train"],  # The prepared training dataset (already tokenized).
    eval_dataset=tokenized_datasets["validation"],  # The validation dataset for evaluation during/after training.
    data_collator=data_collator,  # A function that handles dynamic padding and prepares batches for training.
    tokenizer=tokenizer,  # The tokenizer for padding and other preprocessing tasks.
)

In [ ]:

"""
This output is the result of the trainer.train() call and summarizes the training process.

Note: The configuration now trains for 10 epochs instead of 3, which should improve model performance.

Example metrics you'll see:
- global_step: The total number of training steps (batches) completed.
- training_loss: The average loss of the model over all training steps.
- metrics: A dictionary containing detailed training statistics:
    - 'train_runtime': Total time spent training.
    - 'train_samples_per_second': Training throughput.
    - 'train_steps_per_second': Batches processed per second.
    - 'total_flos': Estimated floating point operations.
    - 'train_loss': Training loss.
    - 'epoch': Number of epochs completed (10.0).

In summary, this output confirms that the model completed 10 epochs of training, providing more learning opportunities for better performance.
"""

trainer.train()

Step,Training Loss
500,0.506000
1000,0.272400


TrainOutput(global_step=1377, training_loss=0.32021946238709603, metrics={'train_runtime': 92.1289, 'train_samples_per_second': 119.441, 'train_steps_per_second': 14.946, 'total_flos': 405114969714960.0, 'train_loss': 0.32021946238709603, 'epoch': 3.0})

In [ ]:
# Use the trained model (via the Trainer object) to make predictions on the validation dataset.
# The 'predict' method runs inference for all samples in the validation set and returns predictions and other relevant data.
predictions = trainer.predict(tokenized_datasets["validation"])

# Display the shapes of the prediction logits and the true label arrays.
# This helps confirm the number of predictions and labels matches the dataset size and the expected model output.
print(predictions.predictions.shape, predictions.label_ids.shape)
# Explanation:
# The output (408, 2) (408,) comes from the shapes of predictions.predictions and predictions.label_ids, respectively.
# - (408, 2): This means predictions.predictions is a NumPy array with 408 rows and 2 columns.
#   Each row contains the raw logits (unnormalized scores) output by the model for the two possible classes, for a total of 408 validation examples.
# - (408,): This means predictions.label_ids is a 1-dimensional array with 408 elements, containing the true class label (0 or 1) for each validation example.
# In summary, both arrays have 408 entries—one for each example in the validation set—so their dimensions align for comparing predictions to true labels.
print("First sample of predictions.predictions:", predictions.predictions[0])
print("First sample of predictions.label_ids:", predictions.label_ids[0])


(408, 2) (408,)
First sample of predictions.predictions: [-3.3215058  3.788661 ]
First sample of predictions.label_ids: 1


In [13]:
# Import the NumPy library, which provides support for numerical operations and array manipulation.
import numpy as np

# Use np.argmax to find the index of the highest logit (score) for each example across the last axis.
# This effectively converts the model's output logits into predicted class labels (0 or 1).
# - predictions.predictions is a 2D array (num_examples, num_classes), containing scores for each class.
# - axis=-1 tells np.argmax to operate along the last axis (i.e., for each example, pick the class with the highest score).
preds = np.argmax(predictions.predictions, axis=-1)
print("First sample of predictions:", preds[0])
print("First sample of true labels:", predictions.label_ids[0])


First sample of predictions: 1
First sample of true labels: 1


In [ ]:
# Import the evaluate library, which provides access to standardized NLP evaluation metrics
import evaluate

# Load the "mrpc" component of the GLUE benchmark for evaluation.
# GLUE ("General Language Understanding Evaluation") is a collection of tasks for evaluating NLP models;
# here, "mrpc" stands for Microsoft Research Paraphrase Corpus, and includes metrics for the paraphrase classification task.
metric = evaluate.load("glue", "mrpc")

# Compute the evaluation metrics (accuracy and F1) by comparing the model's predicted labels (`preds`)
# with the known references (`predictions.label_ids`) from the validation data.
# - `preds` contains the predicted class (0 or 1) for each validation example.
# - `predictions.label_ids` contains the ground-truth class label for each example.
# The result is a dictionary of evaluation scores.
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.8602941176470589, 'f1': 0.9018932874354562}

In [14]:
# This function computes evaluation metrics (accuracy, F1, etc.) for our model during training or evaluation.
def compute_metrics(eval_preds):
    # Load the "mrpc" dataset evaluation metric from the GLUE benchmark suite.
    # This provides accuracy and F1 for paraphrase classification tasks.
    metric = evaluate.load("glue", "mrpc")
    
    # Unpack the tuple returned by the trainer into logits (model outputs) and labels (ground truths).
    logits, labels = eval_preds

    # Convert logits (raw scores for each class) into predicted class labels by taking the index of the highest score.
    # For each example, np.argmax returns 0 or 1 depending on which class has higher score.
    predictions = np.argmax(logits, axis=-1)
    
    # Compute and return the dictionary of evaluation results using the metric,
    # comparing our predicted classes to the true labels.
    return metric.compute(predictions=predictions, references=labels)

In [19]:
# This code sets up and configures the Hugging Face Trainer, which streamlines the process of fine-tuning and evaluating Transformer models.

# 1. Define training arguments:
#    - "test-trainer" defines the output directory for saving model and logs.
#    - 'eval_strategy="epoch"' will evaluate the model at the end of each training epoch.
#    - 'num_train_epochs=10' will train for 10 epochs instead of the default 3.
training_args = TrainingArguments("test-trainer", eval_strategy="epoch", num_train_epochs=10)

# 2. Load a pre-trained sequence classification model (e.g., BERT) for our classification task.
#    - 'checkpoint' specifies which pre-trained model checkpoint to load.
#    - 'num_labels=2' means our task is binary classification (e.g., paraphrase or not).
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# 3. Create a Trainer object that wraps the model, data, training arguments, and evaluation logic.
#    - This handles the training loop, evaluation, and saving models.
trainer = Trainer(
    model,                                   # The model to train
    training_args,                           # Configurations for training/eval/saving/logging
    train_dataset=tokenized_datasets["train"],         # Tokenized training data
    eval_dataset=tokenized_datasets["validation"],     # Tokenized validation data
    data_collator=data_collator,             # Function to form batches (handles dynamic padding)
    tokenizer=tokenizer,                     # The tokenizer for padding and preprocessing
    compute_metrics=compute_metrics,         # Function to compute evaluation metrics (e.g., accuracy, F1)
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_167316/3733406934.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.525736,0.779412,0.850498
2,0.574100,0.437780,0.823529,0.881188
3,0.413200,0.655309,0.833333,0.883959
4,0.281200,0.791801,0.830882,0.880000
5,0.163700,0.868528,0.833333,0.879433
6,0.081200,1.084411,0.825980,0.874780
7,0.034500,1.302169,0.833333,0.885522
8,0.016400,1.190264,0.825980,0.878216
9,0.024100,1.259912,0.833333,0.882759
10,0.013300,1.294783,0.828431,0.880546


TrainOutput(global_step=4590, training_loss=0.1744932917561504, metrics={'train_runtime': 321.2688, 'train_samples_per_second': 114.172, 'train_steps_per_second': 14.287, 'total_flos': 1353042435523440.0, 'train_loss': 0.1744932917561504, 'epoch': 10.0})

### Optimized Hyperparameters for Better Results

Based on the training results, here are improved hyperparameters to potentially enhance performance:


In [22]:
# Optimized hyperparameters for better performance
# Key improvements:
# - Lower learning rate (2e-5) for more stable fine-tuning
# - Warmup steps to gradually increase learning rate
# - More epochs (10) to potentially improve convergence
# - Higher batch size (16) for better gradient estimates
# - Weight decay for regularization to prevent overfitting
# - Learning rate scheduler with warmup
# - Save best model based on evaluation
# - Early stopping patience to prevent overfitting

training_args_optimized = TrainingArguments(
    "test-trainer-optimized",
    num_train_epochs=10,                   # Increase from default 3 to 10
    per_device_train_batch_size=16,        # Set explicit batch size (default is 8)
    per_device_eval_batch_size=16,         # Match eval batch size
    learning_rate=2e-5,                    # Standard learning rate for BERT fine-tuning
    weight_decay=0.01,                     # L2 regularization
    warmup_steps=500,                      # Warm up learning rate over 500 steps
    eval_strategy="epoch",                 # Evaluate after each epoch
    save_strategy="epoch",                 # Save after each epoch
    load_best_model_at_end=True,          # Load best model at the end
    metric_for_best_model="f1",           # Use F1 score as the metric for MRPC (Trainer adds 'eval_' prefix)
    greater_is_better=True,               # Higher is better for F1 score
    logging_steps=100,                     # Log every 100 steps
    seed=42,                              # For reproducibility
    fp16=True,                            # Use mixed precision for faster training
)

model_optimized = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer_optimized = Trainer(
    model_optimized,
    training_args_optimized,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,  # Fixed: using 'tokenizer' instead of 'processing_class'
    compute_metrics=compute_metrics,
)

print("Starting optimized training...")
trainer_optimized.train()

print("\nTraining completed with optimized hyperparameters!")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/tmp/ipykernel_167316/1226491291.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_optimized = Trainer(


Starting optimized training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.593200,0.530954,0.732843,0.824477
2,0.454000,0.424373,0.825980,0.881469
3,0.295700,0.409199,0.823529,0.865672
4,0.196600,0.462145,0.855392,0.897391
5,0.103300,0.682151,0.852941,0.895470
6,0.024100,0.809697,0.843137,0.885714
7,0.033900,0.887989,0.845588,0.891192
8,0.010600,0.926159,0.860294,0.902896
9,0.013700,0.979770,0.860294,0.902896
10,0.005800,0.995774,0.852941,0.896907



Training completed with optimized hyperparameters!


In [23]:
# Make predictions with the optimized model and evaluate
predictions_optimized = trainer_optimized.predict(tokenized_datasets["validation"])
print(f"Optimized model prediction shape: {predictions_optimized.predictions.shape}")

preds_optimized = np.argmax(predictions_optimized.predictions, axis=-1)
results_optimized = metric.compute(predictions=preds_optimized, references=predictions_optimized.label_ids)

print("\n" + "="*60)
print("OPTIMIZED MODEL RESULTS:")
print("="*60)
for key, value in results_optimized.items():
    print(f"{key}: {value:.4f}")
print("="*60)


Optimized model prediction shape: (408, 2)

OPTIMIZED MODEL RESULTS:
accuracy: 0.8603
f1: 0.9029


### Additional Hyperparameter Optimization Suggestions

If you want to experiment further, here are more strategies:

**For Better Accuracy:**
- Increase epochs to 8-10 if not overfitting
- Try learning rates: 1e-5, 3e-5, 5e-5
- Increase batch size to 32 (if GPU memory allows)
- Add gradient accumulation: `gradient_accumulation_steps=2`

**For Faster Training:**
- Use `fp16=True` (already added) or `bf16=True`
- Increase batch size to reduce steps per epoch
- Reduce `logging_steps` for less overhead

**For Stability:**
- Adjust warmup: try 10% of total steps instead of fixed 500
- Use `warmup_ratio=0.1` instead of `warmup_steps`
- Add dropout in model config if overfitting
- Try different optimizers: `optim="adamw_torch"` (default) or `"adafactor"`

**Advanced Techniques:**
- Use `lr_scheduler_type="cosine"` with restarts
- Enable `gradient_checkpointing=True` to save memory
- Try different base models: `"roberta-base"`, `"distilbert-base-uncased"`


In [ ]:
# Save the optimized model and tokenizer locally
model_save_path = "optimized-bert-model"
trainer_optimized.model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved locally to: {model_save_path}")




Model saved locally to: optimized-bert-model


'\nfrom huggingface_hub import notebook_login\n\nnotebook_login()  # This will prompt you for your HF token\n\n# Replace \'your-username\' with your actual HuggingFace username\nhf_repo = "your-username/optimized-bert-model"\n\n# Push the model and tokenizer\ntrainer_optimized.model.push_to_hub(hf_repo)\ntokenizer.push_to_hub(hf_repo)\n\nprint(f"Model pushed to: https://huggingface.co/{hf_repo}")\n'

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token="")
api.upload_folder(
    folder_path="optimized-bert-model",
    repo_id="amelkhoadry/optimized-bert-model",
    repo_type="model",
)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/amelkhoadry/optimized-bert-model/commit/cfde7083fd58ae1d4c00e599b9906e5c73b0dc09', commit_message='Upload folder using huggingface_hub', commit_description='', oid='cfde7083fd58ae1d4c00e599b9906e5c73b0dc09', pr_url=None, repo_url=RepoUrl('https://huggingface.co/amelkhoadry/optimized-bert-model', endpoint='https://huggingface.co', repo_type='model', repo_id='amelkhoadry/optimized-bert-model'), pr_revision=None, pr_num=None)